# 03 — Quasi-Newton Cubic Regularization (L-BFGS)

Compare exact CR, L-BFGS cubic, and plain L-BFGS on high-ish dimensional problems and a tiny MLP.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic, make_synthetic_mlp
from cubic_reg.solvers import cr, quasi_newton
from cubic_reg.plotting import plot_optimality_gap, plot_grad_norm

%matplotlib inline

In [ ]:
p = Quadratic(n=80, condition=200.0, seed=0)
x0 = np.ones(p.dim)
results = {
    "CR exact": cr.minimize(p, x0=x0, M=1.0, eps=1e-8),
    "QN-CR m=5": quasi_newton.minimize(p, x0=x0, M=1.0, eps=1e-6, memory=5),
    "QN-CR m=10": quasi_newton.minimize(p, x0=x0, M=1.0, eps=1e-6, memory=10),
    "QN-CR m=20": quasi_newton.minimize(p, x0=x0, M=1.0, eps=1e-6, memory=20),
    "L-BFGS plain": quasi_newton.minimize_lbfgs_plain(p, x0=x0, eps=1e-6, memory=10),
}
for name, r in results.items():
    print(f"{name:16} nit={r.nit:4d} time={r.time_sec:.4f}s ||g||={r.grad_norm:.2e} f-f*={r.f-p.f_star:.2e}")
plot_optimality_gap(results, f_star=p.f_star, title="Quadratic: exact CR vs L-BFGS cubic")
plt.show()

In [ ]:
mlp = make_synthetic_mlp(n_samples=150, d_in=5, hidden=6, seed=1)
x0 = np.zeros(mlp.dim)
r_qn = quasi_newton.minimize(mlp, x0=x0, M=1.0, eps=1e-4, memory=10, max_iter=80)
r_plain = quasi_newton.minimize_lbfgs_plain(mlp, x0=x0, eps=1e-4, memory=10, max_iter=80)
print("MLP dim", mlp.dim)
print("QN-CR", r_qn.nit, r_qn.f, r_qn.grad_norm, r_qn.time_sec)
print("LBFGS", r_plain.nit, r_plain.f, r_plain.grad_norm, r_plain.time_sec)
plot_grad_norm({"QN-CR": r_qn, "L-BFGS": r_plain}, title="Tiny MLP")
plt.show()